Helicity of segment and CoM

In [ ]:
import gsd.hoomd
import numpy as np
import freud
import math
import matplotlib.pyplot as plt
import seaborn as sns
import csv
from tqdm import tqdm

# ---------------- Helper functions ----------------
def apply_pbc(positions, box):
    Lx, Ly, Lz, xy, xz, yz = box
    box_lengths = np.array([Lx, Ly, Lz])
    return positions - np.floor(positions / box_lengths) * box_lengths

def apply_pbc_distance(r, box):
    Lx, Ly, Lz, xy, xz, yz = box
    box_lengths = np.array([Lx, Ly, Lz])
    r = r - np.round(r / box_lengths) * box_lengths
    return r

# ---------------- Setup chain structure ----------------
sequence = "MGLFDRLGRVVRANLNDLVSKAEDPEKVLEQAVIDMQEDLVQLRQAVARTIAEEKRTEQRLNQDTQEAKKWEDRAKLALTNGEENLAREALARKKSLTDTAAAYQTQLAQQRTMSENLRRNLAALEAKISEAKTKKNMLQARAKAAKANAELQQTLGGLGTSSATSAFERMENKVLDMEATSQAAGELAGFGIENQFAQLEASSGVEDELAALKASMAGGALPGTSAATPQLEAAPVDSSVPANNASQDDAVIDQELDDLRRRLNNL"

group_indices_one_chain = []
particle_index = 0

for amino_acid in sequence:
    beads = 3 if amino_acid == 'G' else 4
    group_indices_one_chain.append(list(range(particle_index, particle_index + beads)))
    particle_index += beads

NUM_CHAINS = 64
particles_per_chain = len([i for sublist in group_indices_one_chain for i in sublist])

def get_group_indices_for_chain(chain_idx, group_indices_one_chain, particles_per_chain):
    offset = chain_idx * particles_per_chain
    return [[idx + offset for idx in residue] for residue in group_indices_one_chain]

def compute_center_of_mass_group(residue_range, positions, masses, box, group_indices_chain):
    range_indices = [
        index
        for res in range(residue_range[0], residue_range[1] + 1)
        for index in group_indices_chain[res]
    ]
    fragment_positions = positions[range_indices]
    fragment_masses = masses[range_indices]

    fragment_positions = apply_pbc(fragment_positions, box)

    total_mass = np.sum(fragment_masses)
    weighted_sum = np.sum(fragment_positions * fragment_masses[:, None], axis=0)

    return weighted_sum / total_mass

bias_values = np.linspace(0.75, 1.0, 12)

# ---------------- Main analysis ----------------
def analyze_data(bias):

    if bias in [0.75, 1.0]:
        trajectory_name = f"trajectory_bias_{bias}.gsd"
    else:
        trajectory_name = f"trajectory_bias_{bias:.16f}.gsd"

    try:
        trajectory = gsd.hoomd.open(
            f"/home/POLY/bhandarit/Desktop/mount_viper/chain_64_IM30/workspace/43ef74dd463bfa6f52acb921ee73e162/{trajectory_name}",
            'r'
        )[10000:20000]
    except FileNotFoundError:
        print(f"Warning: File {trajectory_name} not found. Skipping...")
        return

    all_chains_timeseries = []
    all_chains_timeseries_helix = []

    # ---------------- Dihedral calculation per chain ----------------
    for chain_idx in tqdm(range(NUM_CHAINS), desc=f"Chains bias {bias}"):

        group_indices_chain = get_group_indices_for_chain(
            chain_idx, group_indices_one_chain, particles_per_chain
        )

        table_psi, table_phi = {}, {}

        dihedral_types = trajectory[0].dihedrals.types
        dihedrals_psi = trajectory[0].dihedrals.typeid == dihedral_types.index('psi')
        dihedrals_phi = trajectory[0].dihedrals.typeid == dihedral_types.index('phi')

        for idx, group in enumerate(trajectory[0].dihedrals.group):
            if all(chain_idx * particles_per_chain <= p < (chain_idx + 1) * particles_per_chain for p in group):
                if dihedrals_psi[idx]:
                    table_psi[group[1]] = group
                if dihedrals_phi[idx]:
                    table_phi[group[2]] = group

        mismatched_keys = table_phi.keys() ^ table_psi.keys()
        for key in mismatched_keys:
            table_psi.pop(key, None)
            table_phi.pop(key, None)

        timeseries = {k: np.zeros((len(trajectory), 2)) for k in table_psi}

        def compute_dihedral(array):
            b1 = array[1] - array[0]
            b2 = array[2] - array[1]
            b3 = array[3] - array[2]

            b2 /= np.linalg.norm(b2)

            n1 = np.cross(b1, b2)
            n1 /= np.linalg.norm(n1)

            n2 = np.cross(b2, b3)
            n2 /= np.linalg.norm(n2)

            m1 = np.cross(n1, b2)

            x = np.dot(n1, n2)
            y = np.dot(m1, n2)

            return -np.arctan2(y, x)

        # ---------------- time evolution ----------------
        for time, frame in enumerate(trajectory):

            positions = frame.particles.position
            box = frame.configuration.box

            unwrapped_positions = freud.box.Box.from_box(box).unwrap(
                positions, frame.particles.image
            )

            for key in table_psi.keys():
                psi_group = table_psi[key]
                phi_group = table_phi[key]

                psi_angle = compute_dihedral(unwrapped_positions[psi_group])
                phi_angle = compute_dihedral(unwrapped_positions[phi_group])

                timeseries[key][time] = [phi_angle, psi_angle]

        def is_helical(angles):
            return (
                (angles[:, 0] > math.radians(-160)) &
                (angles[:, 0] < math.radians(-20)) &
                (angles[:, 1] > math.radians(-120)) &
                (angles[:, 1] < math.radians(50))
            )

        timeseries_helix = {}
        keys = list(timeseries.keys())

        for idx, key in enumerate(keys[1:-1]):
            left = keys[idx - 1]
            right = keys[idx + 1]

            helical = (
                is_helical(timeseries[key]) &
                is_helical(timeseries[left]) &
                is_helical(timeseries[right])
            )

            timeseries_helix[idx + 2] = helical

        all_chains_timeseries.append(timeseries)
        all_chains_timeseries_helix.append(timeseries_helix)

    # ---------------- residue fragments ----------------
    residue_ranges = [
        (2, 22), (26, 80), (84, 136),
        (137, 156), (164, 190),
        (192, 215), (251, 264)
    ]

    # store ALL chains
    chain_fragment_propensities = {
        idx: np.zeros((NUM_CHAINS, len(trajectory)))
        for idx in range(len(residue_ranges))
    }

    # ---------------- HELIX PROPENSITY (NO AVERAGING) ----------------
    for chain_idx in range(NUM_CHAINS):
        timeseries_helix = all_chains_timeseries_helix[chain_idx]

        for frag_idx, residue_range in enumerate(residue_ranges):

            fragment_keys = [
                key for key in range(residue_range[0], residue_range[1] + 1)
                if key in timeseries_helix
            ]

            if fragment_keys:
                prop = np.array([
                    np.mean([
                        timeseries_helix[key][time]
                        for key in fragment_keys
                    ])
                    for time in range(len(trajectory))
                ])

                chain_fragment_propensities[frag_idx][chain_idx] = prop

    # ---------------- SAVE HELIX (ALL CHAINS) ----------------
    for frag_idx in chain_fragment_propensities:

        with open(
            f'fragment_{frag_idx}_propensity_bias_{bias:.2f}_all_chains.csv',
            'w',
            newline=''
        ) as file:

            writer = csv.writer(file)

            header = ["Time Frame"] + [f"Chain_{c}" for c in range(NUM_CHAINS)]
            writer.writerow(header)

            for time in range(len(trajectory)):
                row = [time]
                for chain_idx in range(NUM_CHAINS):
                    row.append(chain_fragment_propensities[frag_idx][chain_idx][time])
                writer.writerow(row)

    # ---------------- DISTANCES (ALL CHAINS) ----------------
    chain_fragment_distances = {
        (i, j): np.zeros((NUM_CHAINS, len(trajectory)))
        for i in range(len(residue_ranges))
        for j in range(len(residue_ranges)) if i < j
    }

    for i, range_i in enumerate(residue_ranges):
        for j, range_j in enumerate(residue_ranges):

            if i < j:

                for chain_idx in range(NUM_CHAINS):

                    group_indices_chain = get_group_indices_for_chain(
                        chain_idx, group_indices_one_chain, particles_per_chain
                    )

                    distances = []

                    for frame in trajectory:
                        positions = frame.particles.position
                        masses = frame.particles.mass
                        box = frame.configuration.box

                        com_i = compute_center_of_mass_group(
                            range_i, positions, masses, box, group_indices_chain
                        )
                        com_j = compute_center_of_mass_group(
                            range_j, positions, masses, box, group_indices_chain
                        )

                        dvec = apply_pbc_distance(com_j - com_i, box)
                        distances.append(np.linalg.norm(dvec))

                    chain_fragment_distances[(i, j)][chain_idx] = np.array(distances)

                # ---------------- SAVE ALL CHAINS ----------------
                with open(
                    f'fragment_{i}_{j}_distances_bias_{bias:.2f}_all_chains.csv',
                    'w',
                    newline=''
                ) as file:

                    writer = csv.writer(file)

                    header = ["Time Frame"] + [f"Chain_{c}" for c in range(NUM_CHAINS)]
                    writer.writerow(header)

                    for time in range(len(trajectory)):
                        row = [time]
                        for chain_idx in range(NUM_CHAINS):
                            row.append(chain_fragment_distances[(i, j)][chain_idx][time])
                        writer.writerow(row)

# ---------------- Run ----------------
for bias in bias_values:
    analyze_data(bias)

PCA, Free energy and loading 

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import gaussian_kde
from matplotlib.ticker import MaxNLocator, FuncFormatter

# =============================
# GLOBAL STYLE
# =============================
plt.rcParams.update({
    "font.size": 16,
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
})

def make_ticks_bold():
    plt.xticks(fontweight='bold')
    plt.yticks(fontweight='bold')

def bold_colorbar(cbar, label):
    cbar.set_label(label, fontweight='bold')
    for t in cbar.ax.get_yticklabels():
        t.set_fontweight('bold')

# =============================
# LOAD CHAINS
# =============================
def load_all_chains(path):
    df = pd.read_csv(path)
    chain_cols = [c for c in df.columns if c.startswith("Chain_")]

    if len(chain_cols) == 0:
        raise ValueError(f"No Chain columns in {path}")

    return df[chain_cols].values

# =============================
# SYSTEMS
# =============================
systems = {
    "CHMP3": {
        "dir": "/home/POLY/bhandarit/Desktop/chain_n_chain/PCA_10K_20K/data_64_chain/chmp3/",
        "n_frag": 5
    },
    "IM30": {
        "dir": "/home/POLY/bhandarit/Desktop/chain_n_chain/PCA_10K_20K/data_64_chain/full_length/",
        "n_frag": 7
    },
    "IM30 H0-3": {
        "dir": "/home/POLY/bhandarit/Desktop/chain_n_chain/PCA_10K_20K/data_64_chain/h0-3/",
        "n_frag": 4
    },
    "IM30 H4-6": {
        "dir": "/home/POLY/bhandarit/Desktop/chain_n_chain/PCA_10K_20K/data_64_chain/h4-6/",
        "n_frag": 3
    }
}

save_dir = "PCA_PLOTS"
os.makedirs(save_dir, exist_ok=True)

# =============================
# GLOBAL RANGES
# =============================
global_ranges = {
    "bias": [np.inf, -np.inf],
    "dist": [np.inf, -np.inf],
    "hel": [np.inf, -np.inf],
}

processed_data = {}

# =============================
# PASS 1: BUILD DATA
# =============================
for system, cfg in systems.items():

    print(f"\n[PASS1] Processing {system}")

    base_dir = cfg["dir"]
    n_frag = cfg["n_frag"]

    dist_biases = {
        f.split("bias_")[1].split("_all_chains.csv")[0]
        for f in os.listdir(base_dir)
        if "distances" in f
    }

    hel_biases = {
        f.split("bias_")[1].split("_all_chains.csv")[0]
        for f in os.listdir(base_dir)
        if "propensity" in f
    }

    biases = sorted(dist_biases.intersection(hel_biases))

    all_dist_X, all_hel_X, all_comb_X = [], [], []
    all_bias, all_dist_c, all_hel_c = [], [], []

    for bias in biases:

        dist_list, hel_list = [], []
        ok = True

        for i in range(n_frag):
            for j in range(i + 1, n_frag):
                path = os.path.join(
                    base_dir,
                    f"fragment_{i}_{j}_distances_bias_{bias}_all_chains.csv"
                )
                if not os.path.exists(path):
                    ok = False
                    break
                dist_list.append(load_all_chains(path))
            if not ok:
                break

        if not ok:
            continue

        for i in range(n_frag):
            path = os.path.join(
                base_dir,
                f"fragment_{i}_propensity_bias_{bias}_all_chains.csv"
            )
            if not os.path.exists(path):
                ok = False
                break
            hel_list.append(load_all_chains(path))

        if not ok:
            continue

        min_len = min(x.shape[0] for x in dist_list + hel_list)

        dist_list = [x[:min_len] for x in dist_list]
        hel_list = [x[:min_len] for x in hel_list]

        dist_stack = np.stack(dist_list, axis=-1)
        hel_stack = np.stack(hel_list, axis=-1)

        n_frames, n_chains = dist_stack.shape[0], dist_stack.shape[1]

        dist_X = dist_stack.reshape(n_frames * n_chains, -1)
        hel_X = hel_stack.reshape(n_frames * n_chains, -1)
        comb_X = np.hstack([dist_X, hel_X])

        bias_vals = np.full(n_frames * n_chains, float(bias) * 100)

        all_dist_X.append(dist_X)
        all_hel_X.append(hel_X)
        all_comb_X.append(comb_X)

        all_bias.append(bias_vals)
        all_dist_c.append(np.mean(dist_X, axis=1))
        all_hel_c.append(np.mean(hel_X, axis=1))

    if len(all_dist_X) == 0:
        continue

    dist_X = np.vstack(all_dist_X)
    hel_X = np.vstack(all_hel_X)
    comb_X = np.vstack(all_comb_X)

    bias_c = np.concatenate(all_bias)
    dist_c = np.concatenate(all_dist_c)
    hel_c = np.concatenate(all_hel_c)

    global_ranges["bias"] = [
        min(global_ranges["bias"][0], bias_c.min()),
        max(global_ranges["bias"][1], bias_c.max())
    ]
    global_ranges["dist"] = [
        min(global_ranges["dist"][0], dist_c.min()),
        max(global_ranges["dist"][1], dist_c.max())
    ]
    global_ranges["hel"] = [
        min(global_ranges["hel"][0], hel_c.min()),
        max(global_ranges["hel"][1], hel_c.max())
    ]

    processed_data[system] = (dist_X, hel_X, comb_X, bias_c, dist_c, hel_c)

print("\nGLOBAL RANGES:", global_ranges)

# =============================
# FREE ENERGY
# =============================
def compute_free_energy(x, y):
    kde = gaussian_kde(np.vstack([x, y]))
    xi = np.linspace(x.min(), x.max(), 50)
    yi = np.linspace(y.min(), y.max(), 50)
    Xg, Yg = np.meshgrid(xi, yi)
    Z = kde(np.vstack([Xg.ravel(), Yg.ravel()])).reshape(Xg.shape)
    return Xg, Yg, -np.log(Z + 1e-12)

# =============================
# PCA SCATTER PLOT
# =============================
def plot_pca(X, color, title, cbar_label, filename, vmin, vmax, var):

    fig, ax = plt.subplots(figsize=(6,5))

    sc = ax.scatter(
        X[:,0], X[:,1],
        c=color, cmap="viridis",
        s=10, vmin=vmin, vmax=vmax
    )

    cbar = plt.colorbar(sc)
    bold_colorbar(cbar, cbar_label)

    ax.set_xlabel(f"PC1 ({var[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({var[1]*100:.1f}%)")
    ax.set_title(title)

    ax.xaxis.set_major_locator(MaxNLocator(5))
    ax.yaxis.set_major_locator(MaxNLocator(5))

    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:.1f}"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:.1f}"))

    make_ticks_bold()

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, filename), dpi=600)
    plt.close()

# =============================
# NEW: FEATURE CONTRIBUTION
# =============================
def plot_feature_contribution(pca, n_dist_features, system):

    comps = pca.components_

    dist_loadings = comps[:, :n_dist_features]
    hel_loadings = comps[:, n_dist_features:]

    dist_contrib = np.sum(dist_loadings**2, axis=1)
    hel_contrib = np.sum(hel_loadings**2, axis=1)

    labels = ["PC1", "PC2"]
    x = np.arange(len(labels))
    width = 0.35

    plt.figure(figsize=(6,5))

    plt.bar(x - width/2, dist_contrib, width, label="Distance")
    plt.bar(x + width/2, hel_contrib, width, label="Helicity")

    plt.xticks(x, labels)
    plt.ylabel("Contribution")
    plt.title(f"{system}")

    plt.legend()
    make_ticks_bold()

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{system}_feature_contribution.png"), dpi=600)
    plt.close()

# =============================
# PASS 2: PCA + PLOTS
# =============================
for system, data in processed_data.items():

    print(f"\n[PASS2] Plotting {system}")

    dist_X, hel_X, comb_X, bias_c, dist_c, hel_c = data

    dist_X = StandardScaler().fit_transform(dist_X)
    hel_X = StandardScaler().fit_transform(hel_X)
    comb_X = StandardScaler().fit_transform(comb_X)

    pca = PCA(n_components=2)
    Xc = pca.fit_transform(comb_X)

    var = pca.explained_variance_ratio_

    # NEW: contribution plot
    plot_feature_contribution(
        pca,
        dist_X.shape[1],
        system
    )

    plot_pca(Xc, bias_c, system,
             "H-bond Strength (%)",
             f"{system}_bias.png",
             global_ranges["bias"][0],
             global_ranges["bias"][1],
             var)

    plot_pca(Xc, dist_c, system,
             "Distance (Å)",
             f"{system}_distance.png",
             global_ranges["dist"][0],
             global_ranges["dist"][1],
             var)

    plot_pca(Xc, hel_c, system,
             "Helicity",
             f"{system}_helicity.png",
             global_ranges["hel"][0],
             global_ranges["hel"][1],
             var)

    Xg, Yg, F = compute_free_energy(Xc[:,0], Xc[:,1])

    plt.figure(figsize=(6,5))
    cf = plt.contourf(Xg, Yg, F, levels=20, cmap="coolwarm")

    cbar = plt.colorbar(cf)
    bold_colorbar(cbar, "Free Energy (kT)")

    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title(system)

    make_ticks_bold()
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{system}_FES.png"), dpi=600)
    plt.close()

    print(f"Finished {system}")

print("\n ALL SYSTEMS COMPLETED")
